# Real-time Voice Translator with Eden AI

Two-person live interpretation in one notebook. Person A speaks into the mic → STT → translate to the target language → TTS plays the translated audio back. Switch direction with one click for the reply.

Same pipeline as the [Voice-to-Voice Agent](voice_to_voice_agent.ipynb), but the LLM step is replaced with a translation prompt and we expose a language picker. UN-interpreter style, ~150 lines.

**Prerequisites:** an Eden AI API key, a microphone, and a **browser-based Jupyter frontend** (Classic Notebook or JupyterLab — VS Code's Jupyter extension does not expose mic permissions reliably).

In [ ]:
%pip install --quiet requests ipywebrtc ipywidgets python-dotenv

## 1. Configuration

Swap any of the three model strings to compare providers. The translator uses an LLM (not a dedicated translation endpoint) because LLM translation tends to handle context better than direct MT for short conversational turns.

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_BASE = "https://api.edenai.run/v3"

STT_MODEL = "audio/speech_to_text_async/openai"
LLM_MODEL = "anthropic/claude-sonnet-4-5"
TTS_MODEL = "audio/tts/elevenlabs/eleven_multilingual_v2"  # multilingual TTS — critical for this recipe

LANGUAGES = [
    {"code": "en", "name": "English", "flag": "🇬🇧"},
    {"code": "fr", "name": "French",  "flag": "🇫🇷"},
    {"code": "es", "name": "Spanish", "flag": "🇪🇸"},
    {"code": "de", "name": "German",  "flag": "🇩🇪"},
    {"code": "it", "name": "Italian", "flag": "🇮🇹"},
    {"code": "pt", "name": "Portuguese", "flag": "🇵🇹"},
    {"code": "ja", "name": "Japanese", "flag": "🇯🇵"},
    {"code": "zh", "name": "Chinese", "flag": "🇨🇳"},
    {"code": "ar", "name": "Arabic",  "flag": "🇸🇦"},
]

HEADERS = {"Authorization": f"Bearer {EDENAI_API_KEY}"}
JSON_HEADERS = {**HEADERS, "Content-Type": "application/json"}


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> The audio pipeline does not work on sandbox '
        '(STT receives mocked audio that\'s too small to upload). Use a production key.</div>'
    ))

## 2. Pipeline helpers

Same four helpers as the voice agent — upload, transcribe, translate (replaces chat), synthesize. The translation step is an LLM call with a strict translation prompt (vs. a dedicated translation endpoint) — empirically LLMs handle short conversational turns better.

In [ ]:
import time

import requests


def upload_file(audio_bytes: bytes, filename: str = "recording.webm") -> str:
    r = requests.post(
        f"{EDENAI_BASE}/upload",
        headers=HEADERS,
        files={"file": (filename, audio_bytes, "audio/webm")},
        data={"purpose": "speech"},
    )
    r.raise_for_status()
    return r.json()["file_id"]


def transcribe(file_id: str, language: str) -> str:
    payload = {"model": STT_MODEL, "input": {"file": file_id, "language": language}}
    r = requests.post(f"{EDENAI_BASE}/universal-ai/async", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    data = r.json()
    if data.get("status") == "success":
        return data["output"]["text"]
    job_id = data["public_id"]
    while True:
        time.sleep(1)
        s = requests.get(f"{EDENAI_BASE}/universal-ai/async/{job_id}", headers=HEADERS)
        s.raise_for_status()
        data = s.json()
        if data.get("status") == "success":
            return data["output"]["text"]
        if data.get("status") == "failed":
            raise RuntimeError(data.get("error") or "STT job failed")


def translate(text: str, source_name: str, target_name: str) -> str:
    prompt = (
        f"Translate the following text from {source_name} to {target_name}. "
        "Output only the translation, no quotes, no explanation, no romanization unless the target is Latin-script.\n\n"
        f"TEXT: {text}"
    )
    payload = {"model": LLM_MODEL, "messages": [{"role": "user", "content": prompt}]}
    r = requests.post(f"{EDENAI_BASE}/llm/chat/completions", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


def synthesize(text: str) -> bytes:
    payload = {"model": TTS_MODEL, "input": {"text": text}}
    r = requests.post(f"{EDENAI_BASE}/universal-ai", headers=JSON_HEADERS, json=payload)
    r.raise_for_status()
    audio_url = r.json()["output"]["audio_resource_url"]
    return requests.get(audio_url).content

## 3. UI — two-direction recorder

Pick the two languages (A and B). Click 🎙 A→B to record in language A — STT decodes it as A, translation goes to B, TTS speaks B. Click 🎙 B→A for the reply direction. The transcript accumulates so you can see the conversation.

In [ ]:
from ipywebrtc import AudioRecorder, CameraStream
from ipywidgets import Button, Dropdown, HBox, HTML as HTMLWidget, Layout, Output, VBox
from IPython.display import Audio, clear_output

lang_a_dropdown = Dropdown(
    options=[(f"{l['flag']} {l['name']}", l["code"]) for l in LANGUAGES],
    value="fr",
    description="Person A:",
    layout=Layout(width="280px"),
)
lang_b_dropdown = Dropdown(
    options=[(f"{l['flag']} {l['name']}", l["code"]) for l in LANGUAGES],
    value="en",
    description="Person B:",
    layout=Layout(width="280px"),
)

stream = CameraStream(constraints={"audio": True, "video": False})
recorder = AudioRecorder(stream=stream)

btn_a_to_b = Button(description="▶ A → B", button_style="primary", layout=Layout(width="160px"))
btn_b_to_a = Button(description="◀ B → A", layout=Layout(width="160px"))
btn_reset = Button(description="Reset transcript", layout=Layout(width="160px"))


def _pill(text, color):
    return (
        f'<span style="background:{color};color:white;padding:4px 12px;border-radius:12px;'
        f'font-size:12px;font-family:sans-serif;font-weight:600;">{text}</span>'
    )


status_pill = HTMLWidget(value=_pill("idle", "#6c757d"))
transcript_out = Output()
audio_out = Output()
errors_out = Output()

turns = []  # [{direction: 'a_to_b'|'b_to_a', source, source_text, target, target_text, latencies}]


def _lang_by_code(code):
    return next(l for l in LANGUAGES if l["code"] == code)


def _set_status(text, color):
    status_pill.value = _pill(text, color)


def _render_transcript():
    with transcript_out:
        clear_output()
        for t in turns:
            src = _lang_by_code(t["source"])
            tgt = _lang_by_code(t["target"])
            src_bubble = (
                f'<div style="background:#e3f2fd;padding:8px 12px;border-radius:12px;'
                f'margin:6px 0;max-width:75%;font-family:sans-serif;font-size:13px;">'
                f'<b>{src["flag"]} {src["name"]}</b><br>{t["source_text"]}</div>'
            )
            tgt_bubble = (
                f'<div style="background:#f3e5f5;padding:8px 12px;border-radius:12px;'
                f'margin:6px 0 6px auto;max-width:75%;font-family:sans-serif;font-size:13px;">'
                f'<b>{tgt["flag"]} {tgt["name"]}</b><br>{t["target_text"]}</div>'
            )
            tim = (
                f'<div style="font-size:11px;color:#888;font-family:sans-serif;'
                f'margin-bottom:10px;text-align:right;">'
                f'STT {t["stt_ms"]:.0f}ms · MT {t["mt_ms"]:.0f}ms · TTS {t["tts_ms"]:.0f}ms</div>'
            )
            display(HTML(src_bubble + tgt_bubble + tim))


def run_translation(source_code, target_code):
    errors_out.clear_output()
    audio_bytes = recorder.audio.value
    if not audio_bytes:
        with errors_out:
            display(HTML(
                '<div style="color:#666;font-family:sans-serif;font-size:12px;">'
                '(record something first)</div>'
            ))
        return
    try:
        _set_status("uploading…", "#ffc107")
        file_id = upload_file(audio_bytes)

        _set_status(f"transcribing ({source_code})…", "#17a2b8")
        t0 = time.perf_counter()
        src_text = transcribe(file_id, source_code)
        stt_ms = (time.perf_counter() - t0) * 1000

        src_name = _lang_by_code(source_code)["name"]
        tgt_name = _lang_by_code(target_code)["name"]

        _set_status(f"translating {src_name}→{tgt_name}…", "#007bff")
        t0 = time.perf_counter()
        tgt_text = translate(src_text, src_name, tgt_name)
        mt_ms = (time.perf_counter() - t0) * 1000

        _set_status("synthesizing…", "#6610f2")
        t0 = time.perf_counter()
        audio = synthesize(tgt_text)
        tts_ms = (time.perf_counter() - t0) * 1000

        turns.append({
            "source": source_code, "target": target_code,
            "source_text": src_text, "target_text": tgt_text,
            "stt_ms": stt_ms, "mt_ms": mt_ms, "tts_ms": tts_ms,
        })
        _render_transcript()

        with audio_out:
            clear_output(wait=True)
            display(Audio(audio, autoplay=True))
        _set_status("idle", "#6c757d")
    except Exception as e:
        _set_status("error ✗", "#dc3545")
        with errors_out:
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:12px;'
                f'padding:6px 10px;background:#f8d7da;border-radius:4px;">Error: {e}</div>'
            ))


def on_a_to_b(_):
    run_translation(lang_a_dropdown.value, lang_b_dropdown.value)


def on_b_to_a(_):
    run_translation(lang_b_dropdown.value, lang_a_dropdown.value)


def on_reset(_):
    turns.clear()
    transcript_out.clear_output()
    errors_out.clear_output()
    audio_out.clear_output()
    _set_status("idle", "#6c757d")


btn_a_to_b.on_click(on_a_to_b)
btn_b_to_a.on_click(on_b_to_a)
btn_reset.on_click(on_reset)

controls = HBox([lang_a_dropdown, lang_b_dropdown])
actions = HBox([btn_a_to_b, btn_b_to_a, btn_reset, status_pill])

display(VBox([controls, recorder, actions, transcript_out, audio_out, errors_out]))

## 4. How to use

1. Pick the two languages.
2. Person A speaks: click ● to record, ■ to stop, then **▶ A → B**.
3. The translated audio plays automatically. Both turns appear in the transcript with per-step latencies.
4. Person B replies: record again (the recorder reuses the same mic), then **◀ B → A**.
5. Repeat. **Reset transcript** wipes the conversation history (kept locally in `turns`).

## 5. Customize

**Try different STT providers per language.** OpenAI's STT covers most languages; Deepgram (`audio/speech_to_text_async/deepgram/nova-3`) is faster on English. AssemblyAI is strong on accented English.

**Different TTS voices.** ElevenLabs multilingual is the default because it speaks any of the 9 languages above with a single model. Google `audio/tts/google/wavenet` is cheaper but you need to pick a voice per target language.

**Auto-detect source language.** Drop the `language` param from the STT call (`audio/speech_to_text_async/openai` supports detection) and have the LLM identify which language was spoken in the transcript before translating.

**Preserve speaker voice.** Use a voice-cloning TTS (ElevenLabs supports it) so the translation sounds like the original speaker — closer to a true interpreter UX.